### Set up environment

In [3]:
%pip install transformers tokenizers gradio torch

Note: you may need to restart the kernel to use updated packages.


### import library

In [22]:
import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoImageProcessor, AutoModelForSequenceClassification, AutoModelForImageClassification, pipeline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### Test emotion model

In [20]:
print("Loading emotion tokenizer and model...")
emotion_model_name = "HalogenFlo/microsoft-deberta-v3-base-emotion-recognition"
emotion_tokenizer = AutoTokenizer.from_pretrained(emotion_model_name)
emotion_model = AutoModelForSequenceClassification.from_pretrained(emotion_model_name).to(device)
emotion_labels = ["sadness", "joy", "love", "anger", "fear", "surprise"]

def predict_emotion(text):
    inputs = emotion_tokenizer(text, padding=True, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = emotion_model(**inputs)
    pros = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
    # pred = torch.argmax(pros, dim=-1)
    results = {emotion_labels[i]: float(pros[i]) for i in range(len(emotion_labels))} 
    return dict(sorted(results.items(), key=lambda item: item[1], reverse=True))

print(predict_emotion("I am so happy today!"))

Loading emotion tokenizer and model...


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


{'joy': 0.9996833801269531, 'love': 0.0001246765023097396, 'sadness': 0.00010857565939659253, 'surprise': 6.2257306126412e-05, 'anger': 1.4034575542609673e-05, 'fear': 7.1296331043413375e-06}


### Test vit model

In [35]:
from torchvision import datasets

train_data = datasets.EMNIST(root='./data', split='byclass', train=True, download=True)

In [63]:
print("Loading vit processor and model...")
emnist_model_name = "HalogenFlo/vit-emnist-byclass"
process = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")
emnist_model = AutoModelForImageClassification.from_pretrained(emnist_model_name).to(device)
emnist_labels = [
    '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 
    'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z',
    'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 
    'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z'
]
def predict_character(image):
    if image is None:
        return "Please paint a character or upload image"
    image = image.convert("RGB").resize((224,224))
    inputs = process(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = emnist_model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
    # pred = torch.argmax(probs, dim=-1).item()
    # return f"Label of images is {emnist_labels[pred]} with confidence {probs[pred]}"
    topk_probs, topk_idx = torch.topk(probs, 5)
    results = {}
    for val, idx in zip(topk_probs, topk_idx):
        char_label = emnist_labels[int(idx.item())]
        confidence = float(val.item())
        results[char_label] = confidence
    print(results)
print(predict_character(train_data[4][0]))
train_data[4][0]


Loading vit processor and model...


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


{'M': 0.6868387460708618, 'A': 0.1915910691022873, 'm': 0.026018397882580757, 'R': 0.016094375401735306, 'K': 0.011170442216098309}
None


### Test LLM

In [66]:
print("Loading llm processor and model...")
llm_model_name = "HalogenFlo/qwen-2.5b-finetuned-qlora"
llm_tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
llm_model = AutoModelForCausalLM.from_pretrained(llm_model_name).to(device)

def format_covert(text):
    return f"<|im_start|>user\n{text}\n<|im_end|>\n<|im_start|>assistant"

def generate_text(prompt):
    inputs = llm_tokenizer(format_covert(prompt), return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_length=2048,
            do_sample=False,
            repetition_penalty=1.15,
            eos_token_id=llm_tokenizer.eos_token_id,
            pad_token_id=llm_tokenizer.pad_token_id
        )
    respone = llm_tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:],skip_special_tokens=True)
    print(respone)

generate_text("Truong Sa and Hoang Sa belong?")


Loading llm processor and model...

Vietnamese
